# Pengembangan Model Machine Learning Berbasis Regresi Untuk Valuasi Harga Hunian di Tangerang Selatan

> **Final Project — Study Club Data Science Beginner, KSM Veterantech UPNVJ**

| | |
|---|---|
| **Nama** | Hasan Shofiyyur Rahman |
| **NIM** | 2410512011 |
| **Study Club** | Data Science Beginner — KSM Veterantech UPNVJ |

---

### ML Workflow

```
1. Data Collecting => 2. EDA => 3. Data Preprocessing => 4. Model Training => 5. Model Evaluation
```

# A. Data Collecting 📦

In [ ]:
# Import semua library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimport!")

In [ ]:
# Load dataset
# Path relatif dari folder notebooks/ ke folder data/
DATA_PATH = os.path.join('..', 'data', 'Dataset_HousePricing_South_Tangerang.csv')

df = pd.read_csv(DATA_PATH)
print(f"Dataset berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

In [ ]:
# Informasi dataset
print("Informasi Dataset:")
df.info()

In [ ]:
# Statistik deskriptif
print("\nStatistik Deskriptif:")
df.describe()

# B. Exploratory Data Analysis (EDA) 🔍

In [ ]:
# Pengecekan kolom
print("Kolom yang tersedia:")
print(df.columns.tolist())

In [ ]:
# Pengecekan missing values
print("\nMissing Values per Kolom:")
print(df.isnull().sum())
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

In [ ]:
# Pengecekan duplikat
duplicates = df.duplicated().sum()
print(f"\nJumlah data duplikat: {duplicates}")

# Hapus duplikat
df = df.drop_duplicates()
print(f"Data setelah menghapus duplikat: {df.shape[0]} baris")

# C. Data Preprocessing 🔧

In [ ]:
# Fungsi pembersihan format harga
def clean_price_format(price_str):
    """Membersihkan format harga dari string ke numerik."""
    if pd.isna(price_str):
        return np.nan
    price_str = str(price_str)
    # Hapus 'Rp', spasi, titik sebagai separator ribuan
    price_str = price_str.replace('Rp', '').replace(' ', '').replace('.', '').replace(',', '.')
    try:
        return float(price_str)
    except ValueError:
        return np.nan

# Terapkan pembersihan pada kolom harga (sesuaikan nama kolom dengan dataset)
if 'price' in df.columns:
    df['price_cleaned'] = df['price'].apply(clean_price_format)
elif 'price_cleaned' not in df.columns:
    # Jika kolom sudah bernama price_cleaned, lewati
    print("Kolom harga sudah dalam format bersih.")

print("Kolom setelah preprocessing awal:")
print(df.columns.tolist())

In [ ]:
# Feature Engineering: Extract kecamatan dari listing-location
if 'listing-location' in df.columns:
    df['kecamatan'] = df['listing-location'].str.split(',').str[0].str.strip()
    print("\nKecamatan berhasil di-extract dari listing-location.")

print("\n\nJumlah Area Unik:")
print(df['kecamatan'].nunique())

print("\n\nArea Terbanyak:")
print(df['kecamatan'].value_counts().head(10))

In [ ]:
# Hapus baris dengan missing values pada kolom penting
important_cols = ['bed', 'bath', 'price_cleaned', 'floor_area_sqm', 'kecamatan']
df = df.dropna(subset=[col for col in important_cols if col in df.columns])
print(f"\nData setelah menghapus missing values: {df.shape[0]} baris")

In [ ]:
# Encoding kecamatan
le = LabelEncoder()
df['kecamatan_encoded'] = le.fit_transform(df['kecamatan'])
print("Label Encoding pada kolom kecamatan berhasil.")

In [ ]:
# Korelasi antar fitur
plt.figure(figsize=(10, 8))
correlation = df.corr(numeric_only=True)
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Korelasi Fitur vs Harga Rumah')
plt.tight_layout()
plt.show()

In [ ]:
print("\nKorelasi terhadap Harga:")
print(correlation['price_cleaned'].sort_values(ascending=False))

### Train-Test Split & Scaling

In [ ]:
# Definisikan fitur (X) dan target (y)
X = df[['floor_area_sqm', 'bed', 'bath', 'kecamatan_encoded']]
y = df['price_cleaned']

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Konversi ke DataFrame
X_train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\nSTATUS DATA PREPARATION SELESAI:\n")
print(f"Jumlah Data Latih : {X_train.shape[0]} baris")
print(f"Jumlah Data Uji   : {X_test.shape[0]} baris")
print("\nContoh Data Latih (Scaled):")
X_train_final.head()

# D. Modeling 🤖

In [ ]:
# Model 1: Linear Regression
print("Melatih Model Linear Regression\n")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
print("Model Linear Regression berhasil dilatih!")

In [ ]:
# Model 2: Random Forest Regressor (base)
print("\nMelatih Model Random Forest")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
print("Model Random Forest berhasil dilatih!")

In [ ]:
# Prediksi kedua model
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

# Perbandingan prediksi
comparison = pd.DataFrame({
    'Harga Asli': y_test.values,
    'Tebakan Linear Regression': y_pred_lr,
    'Tebakan Random Forest': y_pred_rf
})

pd.options.display.float_format = '{:,.0f}'.format
print("\nContoh 5 Tebakan Pertama vs Harga Asli")
comparison.head()

# E. Evaluation ⭐

### Evaluasi Model Awal

In [ ]:
print("Evaluasi Model Awal (Random Forest Biasa / Sebelum di-Tuning)")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42)
rf_base.fit(X_train_scaled, y_train)
y_pred_base = rf_base.predict(X_test_scaled)

mae_base = mean_absolute_error(y_test, y_pred_base)
r2_base = r2_score(y_test, y_pred_base)

print(f"MAE Base (Rata-rata Meleset) : Rp {mae_base:,.0f}")
print(f"R2 Base (Akurasi)            : {r2_base:.4f} (Makin dekat 1.0 makin bagus)")

### Hyperparameter Tuning

In [ ]:
print("\nMelakukan Hyperparameter Tuning (Mencari Settingan Terbaik)")

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_tuned = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    verbose=0,
    n_jobs=-1,
    random_state=42
)

rf_tuned.fit(X_train_scaled, y_train)
best_rf_model = rf_tuned.best_estimator_

print(f"Settingan Terbaik Ditemukan: {rf_tuned.best_params_}")

In [ ]:
y_pred_tuned = best_rf_model.predict(X_test_scaled)
print(f"\nMAE Tuned : Rp {mean_absolute_error(y_test, y_pred_tuned):,.0f}")
print(f"R2 Tuned  : {r2_score(y_test, y_pred_tuned):.4f}")

# Cek ada peningkatan?
improvement = r2_score(y_test, y_pred_tuned) - r2_score(y_test, y_pred_base)
print(f"Peningkatan Akurasi: {improvement:.4f} poin")

### Interpretasi Hasil

In [ ]:
# Visualisasi: Harga Asli vs Prediksi
plt.figure(figsize=(10, 6))
y_test_miliar = y_test / 1_000_000_000
y_pred_miliar = y_pred_tuned / 1_000_000_000

sns.scatterplot(x=y_test_miliar, y=y_pred_miliar, alpha=0.6, color='blue', label='Sebaran Prediksi')

min_val = min(y_test_miliar.min(), y_pred_miliar.min())
max_val = max(y_test_miliar.max(), y_pred_miliar.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Garis Sempurna')

plt.title('Seberapa Akurat Tebakan Model? (Harga Asli vs Prediksi)', fontsize=14)
plt.xlabel('Harga Asli (Dalam Miliar Rupiah)', fontsize=12)
plt.ylabel('Harga Prediksi Model (Dalam Miliar Rupiah)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
print("\nInterpretasi: Faktor Penentu Harganya")
importance = best_rf_model.feature_importances_
feature_names = X.columns

fi_df = pd.DataFrame({'Fitur': feature_names, 'Pentingnya': importance})
fi_df = fi_df.sort_values(by='Pentingnya', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Pentingnya', y='Fitur', data=fi_df, palette='viridis')
plt.title('Fitur Apa yang Paling Mempengaruhi Harga Rumah?')
plt.xlabel('Tingkat Kepentingan (0-1)')
plt.ylabel('Fitur')
plt.tight_layout()
plt.show()

---

## Kesimpulan

1. Model **Random Forest Regressor** setelah hyperparameter tuning menghasilkan performa terbaik
2. **Luas bangunan** (`floor_area_sqm`) menjadi fitur yang paling mempengaruhi harga hunian
3. Model mampu menjelaskan ~70% variasi harga hunian (R² ≈ 0.70)
4. Terdapat ruang peningkatan dengan menambahkan fitur tambahan seperti tahun pembangunan, jarak ke fasilitas publik, dll.

---

*Final Project — Study Club Data Science Beginner, KSM Veterantech UPNVJ, 2026*